In [1]:
import pandas as pd

file_main_data = "~/Desktop/number_of_cell_types.csv"
candidate_variables = ['Mean gene length', 'Mean protein length', '#Cell types','Genome size (nt)', '#Genes', '#Protein-coding genes', '#Pfam families', '#Pfam clans', '#Pfam domains', '#Pfam motifs']

## Get data

In [2]:
df = pd.read_csv(file_main_data, sep=",", encoding="utf-8", skiprows=2)
print(df.head().to_string(index=False))
if 1: # Obtener los headers como lista
    headers = df.columns.tolist()
    print(headers)

                          Species  Taxonomy ID  Mean gene length  Mean protein length  Divergency time  #Cell types Source of # of cell types  Genome size (nt)  #Genes  #Protein-coding genes  #Pfam families  #Pfam clans  #Pfam domains  #Pfam motifs       Group Subgroup
                     Homo sapiens         9606       68287.08981           552.845105              0.0        220.0    Vogel and Chothia 2006        3117275501   59444                  20154            3341          493           2774            55 Vertebrates Chordata
                  Pan troglodytes         9598       48873.06647           518.520583              6.4        220.0    Vogel and Chothia 2006        3225356997   44705                  22606            3315          491           2757            55 Vertebrates Chordata
Gallus gallus / Gallus domesticus         9031       30171.87466           558.453461            319.0        190.0    Vogel and Chothia 2006        1053315467   25635                  18023

In [3]:
# Reducir el DataFrame a esas columnas
df_variables = df[candidate_variables]

# show variable df
print(df_variables.head().to_string(index=False))

 Mean gene length  Mean protein length  #Cell types  Genome size (nt)  #Genes  #Protein-coding genes  #Pfam families  #Pfam clans  #Pfam domains  #Pfam motifs
      68287.08981           552.845105        220.0        3117275501   59444                  20154            3341          493           2774            55
      48873.06647           518.520583        220.0        3225356997   44705                  22606            3315          491           2757            55
      30171.87466           558.453461        190.0        1053315467   25635                  18023            2808          488           2626            48
      49193.42265           532.957253        159.0        2728206152   50562                  22192            3284          493           2763            54
      38576.17519           513.326848        159.0        2647899415   42049                  21990            3240          491           2727            52


In [4]:
## Get Spearman correlation

In [5]:
# Calcular la correlación de Spearman
correlacion_spearman = df_variables.corr(method='spearman')

# Mostrar el resultado como tabla
print(correlacion_spearman.to_string())

                       Mean gene length  Mean protein length  #Cell types  Genome size (nt)    #Genes  #Protein-coding genes  #Pfam families  #Pfam clans  #Pfam domains  #Pfam motifs
Mean gene length               1.000000             0.775313     0.927607          0.923926  0.822077               0.750925        0.909907     0.898437       0.908933      0.853047
Mean protein length            0.775313             1.000000     0.636792          0.580731  0.435771               0.379911        0.550871     0.531579       0.547261      0.464747
#Cell types                    0.927607             0.636792     1.000000          0.926802  0.870726               0.825557        0.892124     0.888663       0.893298      0.832745
Genome size (nt)               0.923926             0.580731     0.926802          1.000000  0.945257               0.915492        0.931927     0.919791       0.930866      0.855995
#Genes                         0.822077             0.435771     0.870726          0.

#---
## Calculate partial correlation

Partial correlation formula:

$r_{XY \cdot Z} = \frac{r_{XY} - r_{XZ} \cdot r_{YZ}}{\sqrt{(1 - r_{XZ}^2)(1 - r_{YZ}^2)}}$


In [10]:
import pandas as pd
import numpy as np

# Supongamos que df es tu DataFrame
# df = pd.read_csv("archivo.csv", sep=",", encoding="utf-8", skiprows=2)

x = '#Cell types'
#y = 'Mean gene length'
#y = '#Pfam families'
#y = '#Pfam clans'
y = '#Pfam motifs'

# Lista de columnas numéricas excepto x e y
otras_columnas = [col for col in df_variables.select_dtypes(include=[np.number]).columns if col not in [x, y]]

resultados = {}

for z in otras_columnas:
    # Eliminar filas con NaN en x, y o z
    subset = df_variables[[x, y, z]].dropna()

    # Correlaciones de Spearman entre pares
    r_xy = subset[x].corr(subset[y], method='spearman')
    r_xz = subset[x].corr(subset[z], method='spearman')
    r_yz = subset[y].corr(subset[z], method='spearman')

    # Correlación parcial usando la fórmula
    r_xy_z = (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz**2) * (1 - r_yz**2))

    resultados[z] = r_xy_z

# Mostrar resultados
for z, r in resultados.items():
    print(f"Controlando {z}: correlación parcial de Spearman = {r:.3f}")


Controlando Mean gene length: correlación parcial de Spearman = 0.326
Controlando Mean protein length: correlación parcial de Spearman = 0.778
Controlando Genome size (nt): correlación parcial de Spearman = 0.203
Controlando #Genes: correlación parcial de Spearman = 0.430
Controlando #Protein-coding genes: correlación parcial de Spearman = 0.571
Controlando #Pfam families: correlación parcial de Spearman = -0.199
Controlando #Pfam clans: correlación parcial de Spearman = -0.126
Controlando #Pfam domains: correlación parcial de Spearman = -0.202
